# Presets and comparison with observed graphs

The presets are specifications built from published Slide-recon arrays. This
tutorial draws a few realizations of one of them and compares them with the summary
of the observed graph. Generating a full-size graph takes about half a minute on one
core, so the whole notebook takes a couple of minutes.

In [ ]:
import matplotlib.pyplot as plt

import bipgen

bipgen.presets.available()

In [ ]:
spec = bipgen.presets.load("c58_4")
{k: v for k, v in spec.to_dict().items() if k != "degrees"}

## An ensemble

Three realizations: enough to see the spread, not to estimate tails.

In [ ]:
ensemble = [bipgen.generate(spec, seed=s) for s in range(3)]
for g in ensemble:
    print(g, f"expected / target edges = {g.info['expected_edges'] / g.info['target_edges']:.3f}")

In [ ]:
fig = plt.figure(figsize=(13, 4.6))
bipgen.plot_graph(ensemble[0], ax=fig.add_subplot(1, 3, 1), max_edges=4000, node_size=1)
bipgen.plot_degrees(ensemble[0], axes=[fig.add_subplot(1, 3, 2), fig.add_subplot(1, 3, 3)])
fig.tight_layout()

## Comparison with the observed graph

`presets.observed` holds the statistics and degree sequences of the observed graph,
so the comparison does not need the raw data. The z-score places the observed value
in the ensemble.

In [ ]:
observed = bipgen.presets.observed("c58_4")
table = bipgen.compare(observed, ensemble)
table.round(3)

In [ ]:
bipgen.plot_comparison(table);

The realizations have about 0.4% fewer edges than the observed graph: this is the
loss from clipping probabilities at 1 (`expected_edges` above). Realizations differ
from each other so little that this small gap is many standard deviations. Clipping
also caps the largest degrees, which stay below the observed maxima. Statistics with
no spread in the ensemble, such as the numbers of nodes, have no z-score.

Degree distributions: the distance from the observed degrees, next to the distance
between two realizations, which is what sampling alone produces.

In [ ]:
bipgen.degree_ks(observed, ensemble).round(4)

## A specification from your own graph

`spec_from_graph` takes a biadjacency matrix (BCN as rows, TGT as columns) and reads
off sizes and degree sequences; the kernel scale and the background are chosen by
hand. Here a generated graph stands in for an observed one. With known positions of
some nodes (`positions=`), the density is measured as well and positions are drawn
with the Metropolis method.

In [ ]:
observed_graph = bipgen.generate(bipgen.spec_from_dict({
    "sizes": {"n_bcn": 1500, "n_tgt": 2000},
    "degrees": {"distribution": "lognormal", "bcn": {"median": 12, "sigma": 0.6},
                "tgt": {"sigma": 0.6}},  # shape only: scale set by the BCN total
    "edges": {"model": "kernel", "scale": 0.15},
}), seed=3)
fitted = bipgen.spec_from_graph(observed_graph.biadjacency, scale=0.15)
replicas = [bipgen.generate(fitted, seed=s) for s in range(10)]
bipgen.compare(observed_graph.biadjacency, replicas).round(3)